# gpt-oss-20b 训练全流程

这个 notebook 说明 gpt-oss-20b 的 SFT、PEFT、偏好优化和奖励优化。

gpt-oss-20b 的关键差异是 Harmony 格式、MoE、reasoning effort、MXFP4。训练代码不能简单照搬普通 ChatML 模型。

## 1. 基础配置

- `model_name`：`openai/gpt-oss-20b`。
- `reasoning_effort`：推理强度，常用 `low`、`medium`、`high`。
- `chat_format`：gpt-oss 使用 Harmony，不能直接当普通 ChatML。

In [ ]:
from pathlib import Path

model_name = "openai/gpt-oss-20b"
sft_data_path = "../../data/sample_sft.jsonl"
dpo_data_path = "../../data/sample_preference_dpo.jsonl"
rl_data_path = "../../data/sample_prompts_rl.jsonl"
output_root = Path("../../outputs/gpt_oss_20b")
reasoning_effort = "medium"
output_root.mkdir(parents=True, exist_ok=True)

## 2. Harmony 格式

Harmony 用来表达 system、developer、user、assistant、reasoning 等结构。模板错了，模型可能仍然输出文本，但角色边界和最终回答会不稳定。

后续真实训练时应优先使用官方或框架内置的 gpt-oss chat template，不要手写一个看起来像 ChatML 的模板冒充 Harmony。

In [ ]:
from datasets import load_dataset

sft_dataset = load_dataset("json", data_files=sft_data_path, split="train")
dpo_dataset = load_dataset("json", data_files=dpo_data_path, split="train")
rl_dataset = load_dataset("json", data_files=rl_data_path, split="train")

sft_dataset[0]

## 3. SFT

SFT 用来让 gpt-oss 学习目标回答风格和任务格式。对 gpt-oss 来说，SFT 最大风险不是训练器本身，而是模板错误和精度错误。

建议：

- 优先用 bf16。
- 不建议新人直接全量微调。
- 优先 LoRA/QLoRA 或 Unsloth。
- 先用很小数据验证模板和 loss 是否正常。

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from trl import SFTTrainer

# 会下载并加载模型，确认环境后再执行。
# tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     torch_dtype=torch.bfloat16,
#     device_map="auto",
#     trust_remote_code=True,
# )
# args = TrainingArguments(
#     output_dir=str(output_root / "sft"),
#     per_device_train_batch_size=1,
#     gradient_accumulation_steps=16,
#     learning_rate=1e-5,
#     num_train_epochs=1,
#     logging_steps=1,
#     bf16=True,
#     gradient_checkpointing=True,
#     report_to="none",
# )
# trainer = SFTTrainer(model=model, args=args, train_dataset=sft_dataset, processing_class=tokenizer)
# trainer.train()

## 4. LoRA / QLoRA

gpt-oss-20b 更适合先走 PEFT。LoRA 只训练 adapter，QLoRA 用 4bit 加载基座模型后训练 adapter。

注意：QLoRA 的 bitsandbytes 4bit 和 gpt-oss 权重中的 MXFP4 不是一回事。

In [ ]:
from peft import LoraConfig
from transformers import BitsAndBytesConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# qlora_model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     quantization_config=bnb_config,
#     device_map="auto",
#     trust_remote_code=True,
# )

## 5. DPO

DPO 用 chosen/rejected 偏好数据训练模型。对 gpt-oss 来说，DPO 数据也必须经过正确模板处理，否则偏好学习会建立在错误角色边界上。

In [ ]:
from trl import DPOConfig, DPOTrainer

dpo_config = DPOConfig(
    output_dir=str(output_root / "dpo"),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=5e-7,
    num_train_epochs=1,
    beta=0.1,
    max_prompt_length=1024,
    max_completion_length=1024,
    bf16=True,
    report_to="none",
)

# dpo_trainer = DPOTrainer(model=model, args=dpo_config, train_dataset=dpo_dataset, processing_class=tokenizer)
# dpo_trainer.train()

## 6. PPO / GRPO

PPO 和 GRPO 都需要奖励信号。gpt-oss 的奖励优化要额外关注 reasoning effort：如果训练目标和推理时 effort 设置不一致，效果判断可能不稳定。

GRPO 更适合先从可验证任务开始，例如 exact match、JSON schema、regex match。

In [ ]:
import json
import re

def rule_reward(response: str, answer: str, reward_type: str) -> float:
    text = response.strip()
    if reward_type == "exact_match":
        return 1.0 if text == answer else 0.0
    if reward_type == "regex_match":
        return 1.0 if re.fullmatch(answer.removeprefix("regex:"), text) else 0.0
    if reward_type == "json_schema":
        try:
            return 1.0 if json.loads(text) == json.loads(answer) else 0.0
        except json.JSONDecodeError:
            return 0.0
    raise ValueError(f"unknown reward_type: {reward_type}")

rule_reward("42", rl_dataset[0]["answer"], rl_dataset[0]["reward_type"])

## 7. 衔接关系

- SFT 先验证 Harmony 和 loss。
- LoRA/QLoRA 保存 adapter。
- DPO 需要正确 chosen/rejected 模板。
- GRPO 需要可靠奖励函数。
- 部署前先用 Transformers 或 Ollama 验证 reasoning effort 和输出格式。